# 🏥 AI Health Chatbot — ML Model Analysis

This notebook trains and evaluates all 3 ML models:
1. **Intent Classifier** — TF-IDF + Logistic Regression
2. **Disease Predictor** — Random Forest
3. **Language Detector** — Char N-gram + Naive Bayes


In [ ]:
import os
os.chdir('..')  # go to mlmodels/ root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import cross_val_score

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')
print('✅ Libraries loaded')

## Step 1: Generate Training Data

In [ ]:
exec(open('generate_data.py').read())

## Step 2: Explore Training Data

In [ ]:
df_intent = pd.read_csv('data/intent_training_data.csv')
print('Intent training shape:', df_intent.shape)
df_intent['intent'].value_counts().plot(kind='barh', figsize=(10, 7), color='steelblue')
plt.title('Samples per Intent', fontsize=14, fontweight='bold')
plt.xlabel('# Samples')
plt.tight_layout()
plt.show()

In [ ]:
df_disease = pd.read_csv('data/disease_symptom_data.csv')
print('Disease training shape:', df_disease.shape)
df_disease['disease'].value_counts().plot(kind='bar', color='coral')
plt.title('Samples per Disease', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Symptom co-occurrence heatmap
symptom_cols = [c for c in df_disease.columns if c != 'disease']
corr = df_disease[symptom_cols].corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.5)
plt.title('Symptom Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3: Train All Models

In [ ]:
exec(open('train_models.py').read())

## Step 4: Model Evaluation

In [ ]:
# Load saved models
intent_model   = joblib.load('models/intent_classifier.pkl')
disease_bundle = joblib.load('models/disease_predictor.pkl')
lang_model     = joblib.load('models/lang_detector.pkl')

# Overall accuracy table
results = []

df_i = pd.read_csv('data/intent_training_data.csv')
acc_i = accuracy_score(df_i['intent'], intent_model.predict(df_i['text']))
results.append({'Model': 'Intent Classifier', 'Algorithm': 'TF-IDF + LR', 'Accuracy': f'{acc_i*100:.1f}%'})

df_d = pd.read_csv('data/disease_symptom_data.csv')
Xd = df_d[[c for c in df_d.columns if c != 'disease']]
acc_d = accuracy_score(df_d['disease'], disease_bundle['model'].predict(Xd))
results.append({'Model': 'Disease Predictor', 'Algorithm': 'Random Forest', 'Accuracy': f'{acc_d*100:.1f}%'})

df_l = pd.read_csv('data/language_detection_data.csv')
acc_l = accuracy_score(df_l['language'], lang_model.predict(df_l['text']))
results.append({'Model': 'Language Detector', 'Algorithm': 'Char N-gram + NB', 'Accuracy': f'{acc_l*100:.1f}%'})

pd.DataFrame(results).set_index('Model')

In [ ]:
# Display saved confusion matrix
from IPython.display import Image
Image('plots/intent_confusion_matrix.png')

In [ ]:
# Display feature importances
Image('plots/disease_feature_importance.png')

## Step 5: Live Inference Demo

In [ ]:
test_queries = [
    ('I have high fever and chills after a mosquito bite', 'malaria'),
    ('मलेरिया के लक्षण क्या हैं', 'malaria'),
    ('ଟୀକାକରଣ ସୂଚୀ', 'vaccine'),
    ('dengue fever prevention mosquito', 'dengue'),
    ('tuberculosis tb persistent cough night sweats', 'tuberculosis'),
    ('outbreak alert epidemic spread', 'outbreak_alert'),
    ('vaccine schedule for my baby immunization', 'vaccine'),
    ('emergency ambulance number 108', 'emergency'),
    ('covid-19 symptoms loss of smell', 'covid'),
    ('hello namaste help me', 'greeting'),
]

print(f'{'Query':<45} {'Predicted':<20} {'Expected':<20} {'Match'}')
print('-'*95)
correct = 0
for query, expected in test_queries:
    pred = intent_model.predict([query])[0]
    conf = np.max(intent_model.predict_proba([query])) * 100
    match = '✅' if pred == expected else '❌'
    if pred == expected: correct += 1
    print(f'{query[:43]:<45} {pred:<20} {expected:<20} {match} ({conf:.0f}%)')

print(f'\nDemo Accuracy: {correct}/{len(test_queries)} = {correct/len(test_queries)*100:.0f}%')

In [ ]:
# Language detection demo
lang_tests = [
    ('What are symptoms of malaria?', 'en'),
    ('मुझे बुखार और सिरदर्द है', 'hi'),
    ('ମୋତେ ଜ୍ୱର ଅଛି ଓ ମୁଣ୍ଡ ବ୍ୟଥା', 'or'),
    ('Vaccination for children', 'en'),
    ('टीकाकरण कार्यक्रम', 'hi'),
    ('ଟୀକାକରଣ ସୂଚୀ', 'or'),
]

print(f'{'Text':<40} {'Predicted':<12} {'Expected':<12} {'Match'}')
print('-'*70)
for text, expected in lang_tests:
    pred = lang_model.predict([text])[0]
    conf = np.max(lang_model.predict_proba([text])) * 100
    match = '✅' if pred == expected else '❌'
    print(f'{text[:38]:<40} {pred:<12} {expected:<12} {match} ({conf:.0f}%)')

In [ ]:
# Disease prediction demo
import pandas as pd
symptom_cols = disease_bundle['symptom_cols']
model = disease_bundle['model']

test_cases = [
    (['fever', 'chills', 'headache', 'nausea', 'night_sweats'], 'Malaria'),
    (['fever', 'headache', 'eye_pain', 'rash', 'body_pain', 'bleeding'], 'Dengue'),
    (['cough', 'blood_in_sputum', 'night_sweats', 'weight_loss', 'chest_pain'], 'Tuberculosis'),
    (['diarrhea', 'vomiting', 'dehydration', 'muscle_cramps'], 'Cholera'),
    (['fever', 'cough', 'loss_of_taste', 'loss_of_smell', 'shortness_of_breath'], 'COVID-19'),
]

print(f'{'Symptoms':<50} {'Predicted':<15} {'Expected':<15} {'Match'}')
print('-'*85)
for symptoms, expected in test_cases:
    vec = {s: 0 for s in symptom_cols}
    for s in symptoms: vec[s] = 1
    X = pd.DataFrame([vec])
    pred = model.predict(X)[0]
    conf = np.max(model.predict_proba(X)) * 100
    match = '✅' if pred == expected else '❌'
    syms_str = ', '.join(symptoms[:4]) + ('...' if len(symptoms)>4 else '')
    print(f'{syms_str:<50} {pred:<15} {expected:<15} {match} ({conf:.0f}%)')

## Summary

| Model | Algorithm | Use Case |
|---|---|---|
| Intent Classifier | TF-IDF + Logistic Regression | Maps user message to health intent |
| Disease Predictor | Random Forest | Predicts disease from symptom list |
| Language Detector | Char N-gram + Naive Bayes | Detects EN/HI/OR language |

**Flask API** runs on **port 5001** and is consumed by the Spring Boot backend.
